In [8]:
!pip install chromadb sentence-transformers pandas numpy tqdm torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 30.3 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 46.2 MB/s eta 0:00:001m47.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
# Run this in your local Jupyter notebook
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import chromadb
from sentence_transformers import SentenceTransformer
import torch

# Set your local file paths (UPDATE THESE to match your actual paths)
business_path = '/Users/bhagyapuppala/Downloads/Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json'
review_path = '/Users/bhagyapuppala/Downloads/Yelp JSON/yelp_dataset/yelp_academic_dataset_review.json'

# Verify files exist
print(f"Business file exists: {os.path.exists(business_path)}")
print(f"Review file exists: {os.path.exists(review_path)}")

if os.path.exists(business_path):
    biz_size = os.path.getsize(business_path) / (1024**3)
    rev_size = os.path.getsize(review_path) / (1024**3)
    print(f"Business file size: {biz_size:.2f} GB")
    print(f"Review file size: {rev_size:.2f} GB")



Business file exists: True
Review file exists: True
Business file size: 0.11 GB
Review file size: 4.98 GB


In [11]:
# Let's load only Philadelphia businesses first (much smaller)

print("Loading Philadelphia businesses...")
philly_businesses = []

# Read line by line to save memory
with open(business_path, 'r') as f:
    for line in tqdm(f, desc="Scanning businesses"):
        business = json.loads(line)
        if business.get('city') == 'Philadelphia':
            philly_businesses.append(business)

# Convert to DataFrame
business_df = pd.DataFrame(philly_businesses)
print(f"\n Found {len(business_df)} businesses in Philadelphia")

# Filter to restaurants only
business_df = business_df[business_df['categories'].str.contains('Restaurants', na=False)].copy()
print(f" Found {len(business_df)} restaurants in Philadelphia")

# Show sample
print("\n Sample restaurants:")
print(business_df[['name', 'categories', 'stars', 'review_count']].head())

# Save business IDs for later
philly_business_ids = set(business_df['business_id'].tolist())
print(f"\n Saved {len(philly_business_ids)} unique business IDs")

# Quick stats
print(f"\n Price range distribution:")
business_df['price_range'] = business_df['attributes'].apply(
    lambda x: x.get('RestaurantsPriceRange2') if isinstance(x, dict) else None
)
print(business_df['price_range'].value_counts().sort_index())

Loading Philadelphia businesses...


Scanning businesses: 150346it [00:00, 295171.69it/s]


 Found 14569 businesses in Philadelphia
 Found 5852 restaurants in Philadelphia

 Sample restaurants:
                 name                                         categories  \
0  St Honore Pastries  Restaurants, Food, Bubble Tea, Coffee & Tea, B...   
1            Tuna Bar                  Sushi Bars, Restaurants, Japanese   
2                 BAP                                Korean, Restaurants   
3             Bar One  Cocktail Bars, Bars, Italian, Nightlife, Resta...   
4    DeSandro on Main                    Pizza, Restaurants, Salad, Soup   

   stars  review_count  
0    4.0            80  
1    4.0           245  
2    4.5           205  
3    4.0            65  
4    3.0            41  

 Saved 5852 unique business IDs

 Price range distribution:
price_range
1       2165
2       2485
3        251
4         38
None       1
Name: count, dtype: int64


In [12]:
print("Loading reviews for Philadelphia restaurants...")
philly_reviews = []

# Read review file line by line
with open(review_path, 'r') as f:
    for line in tqdm(f, desc="Loading reviews"):
        review = json.loads(line)
        if review['business_id'] in philly_business_ids:
            philly_reviews.append(review)
            
            # Optional: stop after X reviews if you want to test with smaller sample
            # if len(philly_reviews) >= 50000:  # Uncomment to limit
            #     break

# Convert to DataFrame
review_df = pd.DataFrame(philly_reviews)
print(f"\n✅ Loaded {len(review_df)} reviews")

# Basic stats
print(f"\n📊 Review Statistics:")
print(f"Unique businesses: {review_df['business_id'].nunique()}")
print(f"Date range: {review_df['date'].min()} to {review_df['date'].max()}")
print(f"\nStar distribution:")
print(review_df['stars'].value_counts().sort_index())

# Check text length
review_df['text_length'] = review_df['text'].str.len()
print(f"\n📝 Review text stats:")
print(f"Average length: {review_df['text_length'].mean():.0f} chars")
print(f"Max length: {review_df['text_length'].max()} chars")
print(f"Min length: {review_df['text_length'].min()} chars")

Loading reviews for Philadelphia restaurants...


Loading reviews: 6990280it [00:14, 469760.24it/s]



✅ Loaded 687289 reviews

📊 Review Statistics:
Unique businesses: 5852
Date range: 2005-02-16 04:06:26 to 2022-01-19 19:46:34

Star distribution:
stars
1.0     66624
2.0     57480
3.0     91702
4.0    194366
5.0    277117
Name: count, dtype: int64

📝 Review text stats:
Average length: 602 chars
Max length: 5000 chars
Min length: 1 chars


In [13]:
print("Merging reviews with business data...")

# Merge review and business data
merged_df = review_df.merge(
    business_df[[
        'business_id', 'name', 'categories', 'stars', 
        'attributes', 'review_count', 'is_open'
    ]],
    on='business_id',
    how='left',
    suffixes=('_review', '_business')
)

print(f"✅ Merged shape: {merged_df.shape}")

# Check for missing values
print("\n🔍 Missing values:")
print(merged_df.isnull().sum())

# Create clean text column
print("\n🧹 Cleaning text...")
merged_df['clean_text'] = merged_df['text'].apply(
    lambda x: ' '.join(str(x).split())  # Remove extra whitespace
)

# Extract basic metadata from attributes
def extract_basic_attrs(attrs):
    """Extract only the most essential attributes"""
    if pd.isna(attrs) or not isinstance(attrs, dict):
        return {
            'price_range': None,
            'alcohol': None,
            'noise_level': None
        }
    
    return {
        'price_range': attrs.get('RestaurantsPriceRange2'),
        'alcohol': attrs.get('Alcohol'),
        'noise_level': attrs.get('NoiseLevel')
    }

print("📊 Extracting attributes...")
attrs_df = pd.json_normalize(merged_df['attributes'].apply(extract_basic_attrs))
merged_df = pd.concat([merged_df, attrs_df], axis=1)

# Extract year from date for filtering
merged_df['year'] = pd.to_datetime(merged_df['date']).dt.year

print("\n✅ Final columns:")
print(merged_df.columns.tolist())

# Show sample
print("\n📝 Sample data:")
print(merged_df[['review_id', 'name', 'stars_review', 'clean_text', 'price_range']].head())

Merging reviews with business data...
✅ Merged shape: (687289, 16)

🔍 Missing values:
review_id           0
user_id             0
business_id         0
stars_review        0
useful              0
funny               0
cool                0
text                0
date                0
text_length         0
name                0
categories          0
stars_business      0
attributes        388
review_count        0
is_open             0
dtype: int64

🧹 Cleaning text...
📊 Extracting attributes...

✅ Final columns:
['review_id', 'user_id', 'business_id', 'stars_review', 'useful', 'funny', 'cool', 'text', 'date', 'text_length', 'name', 'categories', 'stars_business', 'attributes', 'review_count', 'is_open', 'clean_text', 'price_range', 'alcohol', 'noise_level', 'year']

📝 Sample data:
                review_id               name  stars_review  \
0  AqPFMleE6RsU23_auESxiA              Zaika           5.0   
1  JrIxlS1TzJ-iCu79ul40cQ           Dmitri's           1.0   
2  8JFGBuHMoiNDyfcxuWNtr

In [14]:
merged_df

,review_id,user_id,business_id,stars_review,useful,funny,cool,text,date,text_length,...,categories,stars_business,attributes,review_count,is_open,clean_text,price_range,alcohol,noise_level,year
0,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,243,...,"Halal, Pakistani, Restaurants, Indian",4.0,"{'Caters': 'True', 'Ambience': '{'romantic': F...",181,1,"Wow! Yummy, different, delicious. Our favorite...",2,u'none',u'average',2015
1,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1.0,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,341,...,"Mediterranean, Restaurants, Seafood, Greek",4.0,"{'BusinessParking': '{'garage': False, 'street...",273,0,I am a long term frequent customer of this est...,2,'none',u'average',2015
2,8JFGBuHMoiNDyfcxuWNtrA,smOvOajNG0lS4Pq7d8g4JQ,RZtGWDLCAtuipwaZ-UfjmQ,4.0,0,0,0,Good food--loved the gnocchi with marinara\nth...,2009-10-14 19:57:14,175,...,"Pizza, Restaurants, Italian, Salad",3.5,"{'RestaurantsReservations': 'True', 'BYOBCorka...",367,0,Good food--loved the gnocchi with marinara the...,2,u'full_bar',u'average',2009
3,oyaMhzBSwfGgemSGuZCdwQ,Dd1jQj7S-BFGqRbApFzCFw,YtSqYv1Q_pOltsVPSx54SA,5.0,0,0,0,Tremendous service (Big shout out to Douglas) ...,2013-06-24 11:21:25,276,...,"Wine Bars, Restaurants, Nightlife, Steakhouses...",3.5,"{'RestaurantsAttire': 'u'dressy'', 'Restaurant...",290,1,Tremendous service (Big shout out to Douglas) ...,3,'full_bar',u'average',2013
4,Xs8Z8lmKkosqW5mw_sVAoA,IQsF3Rc6IgCzjVV9DE8KXg,eFvzHawVJofxSnD7TgbZtg,5.0,0,0,0,My absolute favorite cafe in the city. Their b...,2014-11-12 15:30:27,419,...,"Food, Cafes, Coffee & Tea, Restaurants",4.0,"{'Alcohol': 'u'none'', 'RestaurantsReservation...",249,1,My absolute favorite cafe in the city. Their b...,1,u'none',u'average',2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687284,Wrt6pZizQzw-ZTKrvMwrGw,ua6QuBe6mar6pDrhHETzJQ,Bk_1vsPtOtO0bojfQZQIOw,2.0,3,3,2,"Ok, I have to say, after living on this block ...",2008-12-15 23:18:27,1325,...,"Sushi Bars, Restaurants",3.5,"{'RestaurantsTakeOut': 'True', 'OutdoorSeating...",40,0,"Ok, I have to say, after living on this block ...",2,'none',u'quiet',2008
687285,X5R98ygOtbhryDiKA-J2qQ,LHWtjTG7e1NzNPYUbUo-9w,rgeuy1qbw6Z8B6CSVANHIA,5.0,1,1,1,I've been to the other Federal Donuts location...,2012-10-13 14:39:37,752,...,"Donuts, Sandwiches, Soul Food, Food, Coffee & ...",4.0,"{'RestaurantsReservations': 'False', 'Alcohol'...",1464,0,I've been to the other Federal Donuts location...,1,u'none',u'average',2012
687286,MVg4YUQeEhCA7Z7RsBJSVg,7-7A0Avj47slLGV7yBFc8w,ytynqOUb3hjKeJfRj5Tshw,3.0,1,0,0,"I was so excited about all the food I saw, but...",2013-07-25 21:00:15,262,...,"Candy Stores, Shopping, Department Stores, Fas...",4.5,"{'RestaurantsGoodForGroups': 'True', 'Restaura...",5721,1,"I was so excited about all the food I saw, but...",2,'beer_and_wine',u'loud',2013
687287,nLjbVsETpqO17RbFcqskkA,am7-gkH_PDz598oTdYSD6A,3gVSrS4kffGGZT8oXHsIcw,3.0,2,0,2,"*Later Yelp* I've only been here once, but I l...",2014-11-03 14:45:46,361,...,"Restaurants, Lounges, Asian Fusion, Nightlife,...",3.5,"{'RestaurantsReservations': 'True', 'Restauran...",797,1,"*Later Yelp* I've only been here once, but I l...",2,'full_bar',u'average',2014


In [15]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB (persistent storage)
chroma_client = chromadb.PersistentClient(
    path="./yelp_chromadb"  # Saves to your local folder
)

# Create or get collection
collection = chroma_client.get_or_create_collection(
    name="philadelphia_restaurant_reviews",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

print(f"✅ ChromaDB collection ready: {collection.name}")

# Check if collection is empty
count = collection.count()
print(f"Current documents in collection: {count}")

# If you want to delete and start fresh (uncomment only if needed)
# chroma_client.delete_collection("philadelphia_restaurant_reviews")
# print("♻️ Collection deleted - run chunk again to recreate")

✅ ChromaDB collection ready: philadelphia_restaurant_reviews
Current documents in collection: 0


In [16]:
from sentence_transformers import SentenceTransformer
import torch

# Load model (384-dim embeddings - good balance of speed/quality)
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"✅ Model loaded on {device}")

# Take a small test batch first (100 reviews)
test_df = merged_df.head(100).copy()

# Generate test embeddings
print("\n🧪 Testing with 100 reviews...")
test_embeddings = model.encode(
    test_df['clean_text'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"✅ Test embeddings shape: {test_embeddings.shape}")
print(f"Sample embedding (first 5 numbers): {test_embeddings[0][:5]}")

# Prepare metadata for test batch (MINIMAL - just IDs for now)
test_metadatas = []
for _, row in test_df.iterrows():
    test_metadatas.append({
        'review_id': row['review_id'],
        'business_id': row['business_id'],
        'rating': float(row['stars_review']),
        'year': int(row['year']) if pd.notna(row['year']) else 0,
        'price_range': int(row['price_range']) if pd.notna(row['price_range']) else 0
    })

print(f"\n✅ Test metadata prepared")
print(f"Sample metadata: {test_metadatas[0]}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded on cpu

🧪 Testing with 100 reviews...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Test embeddings shape: (100, 384)
Sample embedding (first 5 numbers): [-0.08979224 -0.04855577  0.005521    0.00049564 -0.05056455]

✅ Test metadata prepared
Sample metadata: {'review_id': 'AqPFMleE6RsU23_auESxiA', 'business_id': 'kxX2SOes4o-D3ZQBkiMRfA', 'rating': 5.0, 'year': 2015, 'price_range': 2}


In [18]:
import time

# Use the full merged_df (687k reviews)
full_df = merged_df.copy()

# Generate unique IDs for ChromaDB (use review_id)
ids = full_df['review_id'].tolist()

# Prepare documents (the text to embed)
documents = full_df['clean_text'].tolist()

# Prepare minimal metadata (fixed to handle string 'None')
print("Preparing metadata for all reviews...")
metadatas = []
for _, row in tqdm(full_df.iterrows(), total=len(full_df), desc="Creating metadata"):
    
    # Handle price_range (could be string 'None', actual None, or number)
    price = row['price_range']
    if pd.isna(price) or price == 'None' or price is None:
        price_val = 0
    else:
        try:
            price_val = int(float(price))  # Convert to float first then int
        except:
            price_val = 0
    
    # Handle year
    year = row['year']
    if pd.isna(year) or year == 'None' or year is None:
        year_val = 0
    else:
        try:
            year_val = int(float(year))
        except:
            year_val = 0
    
    metadatas.append({
        'review_id': str(row['review_id']),
        'business_id': str(row['business_id']),
        'rating': float(row['stars_review']) if pd.notna(row['stars_review']) else 0.0,
        'year': year_val,
        'price_range': price_val,
        'restaurant_name': str(row['name'])[:50]  # Limit length
    })

print(f"✅ Prepared {len(metadatas)} metadata entries")

# Process in batches (adjust batch_size based on your RAM)
batch_size = 1000  # You can increase to 2000/5000 if you have enough RAM
total_batches = (len(full_df) + batch_size - 1) // batch_size

print(f"\n🚀 Starting embedding of {len(full_df)} reviews in {total_batches} batches...")
print(f"Batch size: {batch_size}")

start_time = time.time()

for i in range(0, len(full_df), batch_size):
    batch_num = i//batch_size + 1
    batch_end = min(i + batch_size, len(full_df))
    
    print(f"\n📦 Batch {batch_num}/{total_batches} (reviews {i} to {batch_end})")
    
    # Get batch data
    batch_texts = documents[i:batch_end]
    batch_ids = ids[i:batch_end]
    batch_metadatas = metadatas[i:batch_end]
    
    # Generate embeddings for this batch
    batch_embeddings = model.encode(
        batch_texts,
        show_progress_bar=True,
        batch_size=32  # Internal batch size for the model
    ).tolist()  # Convert to list for ChromaDB
    
    # Add to ChromaDB
    collection.add(
        embeddings=batch_embeddings,
        documents=batch_texts,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
    
    print(f"✅ Batch {batch_num} complete. Total in DB: {collection.count()}")
    
    # Optional: small pause between batches
    time.sleep(1)

elapsed = time.time() - start_time
print(f"\n🎉 DONE! Added {collection.count()} reviews to ChromaDB")
print(f"Time taken: {elapsed/60:.2f} minutes")

Preparing metadata for all reviews...


Creating metadata: 100%|█████████████| 687289/687289 [00:08<00:00, 79666.98it/s]

✅ Prepared 687289 metadata entries

🚀 Starting embedding of 687289 reviews in 688 batches...
Batch size: 1000

📦 Batch 1/688 (reviews 0 to 1000)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

✅ Batch 1 complete. Total in DB: 1000

📦 Batch 2/688 (reviews 1000 to 2000)


...

✅ Batch 686 complete. Total in DB: 686000

📦 Batch 687/688 (reviews 686000 to 687000)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

✅ Batch 687 complete. Total in DB: 687000

📦 Batch 688/688 (reviews 687000 to 687289)


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Batch 688 complete. Total in DB: 687289

🎉 DONE! Added 687289 reviews to ChromaDB
Time taken: 101.51 minutes


In [19]:
# Check collection stats
print(f"📊 Collection name: {collection.name}")
print(f"📊 Total documents: {collection.count()}")

# Get a sample to verify structure
sample = collection.get(limit=5)
print("\n📝 Sample IDs:")
print(sample['ids'])

print("\n📝 Sample metadata (first 2):")
for i, meta in enumerate(sample['metadatas'][:2]):
    print(f"\nReview {i+1}:")
    for key, value in meta.items():
        print(f"  {key}: {value}")

print("\n📝 Sample document preview (first 2):")
for i, doc in enumerate(sample['documents'][:2]):
    print(f"\nReview {i+1}: {doc[:100]}...")

# Test a simple query
print("\n🔍 Testing search for 'pizza'...")
results = collection.query(
    query_texts=["great pizza"],
    n_results=5
)

print("\n✅ Top 5 results:")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"\n{i+1}. {meta['restaurant_name']} (Rating: {meta['rating']}⭐, Price: {'$' * meta['price_range']})")
    print(f"   Distance: {dist:.3f}")
    print(f"   Preview: {doc[:150]}...")

📊 Collection name: philadelphia_restaurant_reviews
📊 Total documents: 687289

📝 Sample IDs:
['AqPFMleE6RsU23_auESxiA', 'JrIxlS1TzJ-iCu79ul40cQ', '8JFGBuHMoiNDyfcxuWNtrA', 'oyaMhzBSwfGgemSGuZCdwQ', 'Xs8Z8lmKkosqW5mw_sVAoA']

📝 Sample metadata (first 2):

Review 1:
  year: 2015
  review_id: AqPFMleE6RsU23_auESxiA
  price_range: 2
  restaurant_name: Zaika
  business_id: kxX2SOes4o-D3ZQBkiMRfA
  rating: 5.0

Review 2:
  price_range: 2
  year: 2015
  business_id: 04UD14gamNjLY0IDYVhHJg
  restaurant_name: Dmitri's
  review_id: JrIxlS1TzJ-iCu79ul40cQ
  rating: 1.0

📝 Sample document preview (first 2):

Review 1: Wow! Yummy, different, delicious. Our favorite is the lamb curry and korma. With 10 different kinds ...

Review 2: I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) ...

🔍 Testing search for 'pizza'...


/Users/bhagyapuppala/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100



✅ Top 5 results:

1. Slice (Rating: 5.0⭐, Price: $$)
   Distance: 0.102
   Preview: Good pizza...

2. Charlie's Pizzeria (Rating: 5.0⭐, Price: $)
   Distance: 0.235
   Preview: Great pizzas This pizza is the best pizza I've ever had in my entire living. Nice and hot. cheese and sauce was to perfection. We had pepperoni and sa...

3. SOHO Pizza (Rating: 5.0⭐, Price: $)
   Distance: 0.242
   Preview: Amazing pizza!! Awesome. Just love it. If you want great pizza, go here. End of story....

4. Mack's Boardwalk Pizza (Rating: 5.0⭐, Price: $)
   Distance: 0.248
   Preview: Great pizza great entertainment great value!!!...

5. Zuzu's Kitchen (Rating: 5.0⭐, Price: $)
   Distance: 0.250
   Preview: Great pizza, outstanding service!! Their speciality pizzas are wonderful. Very quick and served fresh....


In [20]:
# Create a unique business list with all metadata for community detection
business_list = merged_df[[
    'business_id', 'name', 'categories', 'price_range', 
    'alcohol', 'noise_level', 'stars_business', 'review_count'
]].drop_duplicates('business_id')

print(f"📊 Unique businesses: {len(business_list)}")

# Save to CSV for Person 1-2 (community detection)
business_list.to_csv('philly_restaurants_for_community.csv', index=False)
print("✅ Saved: philly_restaurants_for_community.csv")

# Also save a simple business_id to name mapping for Person 4
business_list[['business_id', 'name', 'categories']].to_csv(
    'business_id_mapping.csv', index=False
)
print("✅ Saved: business_id_mapping.csv")

print("\n📁 Files ready for:")
print("- Person 1-2: Use 'philly_restaurants_for_community.csv' for community detection")
print("- Person 4: Use 'business_id_mapping.csv' to look up restaurant names")
print("- Person 4: ChromaDB collection 'philadelphia_restaurant_reviews' is ready!")

# Quick test to show Person 4 how to query
print("\n🔍 Example query for Person 4:")
example = collection.query(
    query_texts=["romantic dinner"],
    n_results=3,
    where={"price_range": 3}  # Filter by price
)
print(f"Found {len(example['ids'][0])} results for 'romantic dinner' with price_range=3")

📊 Unique businesses: 5852
✅ Saved: philly_restaurants_for_community.csv
✅ Saved: business_id_mapping.csv

📁 Files ready for:
- Person 1-2: Use 'philly_restaurants_for_community.csv' for community detection
- Person 4: Use 'business_id_mapping.csv' to look up restaurant names
- Person 4: ChromaDB collection 'philadelphia_restaurant_reviews' is ready!

🔍 Example query for Person 4:
Found 3 results for 'romantic dinner' with price_range=3


In [23]:
import pandas as pd
import chromadb

# Load the community mapping file
community_df = pd.read_excel('/Users/bhagyapuppala/Downloads/business_community_map_with_names.xlsx')
print(f"✅ Loaded {len(community_df)} business-community mappings")
print(community_df.head())

# Connect to your existing ChromaDB
chroma_client = chromadb.PersistentClient(path="./yelp_chromadb")
collection = chroma_client.get_collection("philadelphia_restaurant_reviews")

print(f"\n📊 Current documents in DB: {collection.count()}")

# Process each business from the mapping file directly
print("\n🔄 Updating metadata business by business...")
updated_count = 0
businesses_updated = 0

for idx, row in community_df.iterrows():
    business_id = row['business_id']
    community_id = int(row['community_id'])
    
    try:
        # Get ALL reviews for THIS SPECIFIC business only (much smaller query)
        business_reviews = collection.get(
            where={"business_id": business_id},
            include=['metadatas']
        )
        
        review_ids = business_reviews['ids']
        
        if len(review_ids) > 0:
            # Get the current metadata from first review
            current_metadata = business_reviews['metadatas'][0]
            
            # Add community_id to metadata
            new_metadata = current_metadata.copy()
            new_metadata['community_id'] = community_id
            
            # Update ALL reviews for this business
            collection.update(
                ids=review_ids,
                metadatas=[new_metadata] * len(review_ids)
            )
            
            updated_count += len(review_ids)
            businesses_updated += 1
            print(f"  ✅ {idx+1}/{len(community_df)}: Updated {len(review_ids)} reviews for {business_id[:8]}... (community {community_id})")
            
    except Exception as e:
        print(f"  ❌ Error with business {business_id[:8]}...: {str(e)[:100]}")
        continue

print(f"\n🎉 DONE! Updated {updated_count} reviews across {businesses_updated} businesses")

# Verify the update worked
print("\n🔍 Verifying sample update:")
sample = collection.get(limit=5)
for i, meta in enumerate(sample['metadatas']):
    community = meta.get('community_id', 'NOT SET')
    print(f"Review {i+1} - Business: {meta['business_id'][:8]}..., Community: {community}")

print("\n✅ Ready for Person 4!")

✅ Loaded 499 business-community mappings
              business_id            business_name  community_id
0  ytynqOUb3hjKeJfRj5Tshw  Reading Terminal Market             1
1  PP3BBaVxZLcJU54uP_wL6Q     Pat's King of Steaks             3
2  IkY2ticzHEn4QFn8hQLSWg            Geno's Steaks             0
3  9PZxjhTIU7OgPIzuGi89Ew                   El Vez             0
4  ctHjyadbDQAtUFfkcAFEHw                    Zahav             0

📊 Current documents in DB: 687289

🔄 Updating metadata business by business...
  ❌ Error with business ytynqOUb...: ValueError: Batch size of 5778 is greater than max batch size of 5461
  ✅ 2/499: Updated 4293 reviews for PP3BBaVx... (community 3)
  ✅ 3/499: Updated 3428 reviews for IkY2ticz... (community 0)
  ✅ 4/499: Updated 3264 reviews for 9PZxjhTI... (community 0)
  ✅ 5/499: Updated 3173 reviews for ctHjyadb... (community 0)
  ✅ 6/499: Updated 2974 reviews for 6ajnOk0G... (community 0)
  ✅ 7/499: Updated 2884 reviews for j-qtdD55... (community 4)
  ✅ 8/499: